# Predicciones — Blind Test Data
## ML Modeling Challenge - Wizeline

Generación de predicciones sobre `blind_test_data.csv` usando el modelo **CatBoost optimizado**
guardado en el Notebook 3.

- Modelo  : cargado desde `models/catboost_optimized.joblib`
- Features : leídas desde `config.yaml`
- Output  : `data/predictions.csv`

## 0. Imports y configuración

In [1]:
from pathlib import Path

import joblib
import plotly.express as px
import polars as pl
import yaml

with open('config.yaml') as f:
    config = yaml.safe_load(f)

FEATURES: list[str] = config['features']['selected']
MODEL_PATH: Path = Path(config['model']['artifact_path'])
BLIND_TEST_PATH: Path = Path(config['data']['blind_test_path'])
OUTPUT_PATH: Path = Path('data/predictions.csv')

print(f'Modelo          : {MODEL_PATH}')
print(f'Features ({len(FEATURES)})   : {FEATURES}')
print(f'Blind test path : {BLIND_TEST_PATH}')
print(f'Output path     : {OUTPUT_PATH}')


Modelo          : models/catboost_optimized.joblib
Features (5)   : ['feature_2', 'feature_13', 'feature_9', 'feature_18', 'feature_11']
Blind test path : data/blind_test_data.csv
Output path     : data/predictions.csv


## 1. Carga del modelo y datos

In [2]:
pipeline = joblib.load(MODEL_PATH)
print(f'Modelo cargado: {type(pipeline.named_steps["model"]).__name__}')
print(f'Parámetros del estimador:')
for k, v in pipeline.named_steps['model'].get_params().items():
    print(f'  {k}: {v}')


Modelo cargado: CatBoostRegressor
Parámetros del estimador:
  iterations: 980
  learning_rate: 0.038518088621849036
  depth: 5
  l2_leaf_reg: 9.56569213287667
  loss_function: RMSE
  random_seed: 5000
  verbose: 0
  subsample: 0.8000817293258272


In [3]:
df_blind = pl.read_csv(BLIND_TEST_PATH)
print(f'Shape blind test : {df_blind.shape}')
print(f'Columnas         : {df_blind.columns}')
df_blind.head()


Shape blind test : (200, 20)
Columnas         : ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19']


feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
676.867615,32.518822,254.825875,502.26851,609.469688,497.624266,105.246239,269.045539,150.177005,312.64986,765.296227,0.237996,660.030637,147.059794,40.232132,464.424834,121.147466,68.284243,26.96987,314.461582
628.695228,426.163933,347.07028,431.106903,915.527507,301.699534,1.666992,306.733041,104.234252,63.24207,467.009734,6.608084,600.994184,43.619815,48.153926,457.256565,49.163652,85.511662,33.500538,819.537877
131.765943,323.839669,245.399775,181.814398,710.179159,59.117377,312.622788,687.965027,109.803179,381.1695,700.532108,1.82237,736.306092,138.759029,36.915389,436.174065,10.037994,62.631938,6.211169,341.361374
160.970195,489.712029,70.482159,309.486269,888.030604,412.655666,216.124989,47.415477,104.139145,326.462385,378.446187,1.686895,485.144327,143.668518,27.168148,309.715497,149.661493,66.415878,15.001753,539.087409
419.907137,216.625219,487.88786,253.704462,323.226862,65.744463,271.811469,527.726782,129.805782,168.429679,637.944633,0.948507,365.946758,72.337904,36.232169,302.772338,186.944884,106.514846,3.443809,364.341969


## 2. Generación de predicciones

In [4]:
X_blind = df_blind.select(FEATURES).to_numpy()
predictions = pipeline.predict(X_blind)

df_predictions = df_blind.with_columns(
    pl.Series('target_predicted', predictions)
)

print(f'Predicciones generadas: {len(predictions)}')
print(f'\nEstadísticas de las predicciones:')
print(df_predictions.select('target_predicted').describe())
df_predictions.head()


Predicciones generadas: 200

Estadísticas de las predicciones:
shape: (9, 2)
┌────────────┬──────────────────┐
│ statistic  ┆ target_predicted │
│ ---        ┆ ---              │
│ str        ┆ f64              │
╞════════════╪══════════════════╡
│ count      ┆ 200.0            │
│ null_count ┆ 0.0              │
│ mean       ┆ 14.223496        │
│ std        ┆ 4.961928         │
│ min        ┆ 2.019092         │
│ 25%        ┆ 10.422584        │
│ 50%        ┆ 14.258765        │
│ 75%        ┆ 17.671019        │
│ max        ┆ 25.427945        │
└────────────┴──────────────────┘


feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,target_predicted
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
676.867615,32.518822,254.825875,502.26851,609.469688,497.624266,105.246239,269.045539,150.177005,312.64986,765.296227,0.237996,660.030637,147.059794,40.232132,464.424834,121.147466,68.284243,26.96987,314.461582,13.025063
628.695228,426.163933,347.07028,431.106903,915.527507,301.699534,1.666992,306.733041,104.234252,63.24207,467.009734,6.608084,600.994184,43.619815,48.153926,457.256565,49.163652,85.511662,33.500538,819.537877,11.719347
131.765943,323.839669,245.399775,181.814398,710.179159,59.117377,312.622788,687.965027,109.803179,381.1695,700.532108,1.82237,736.306092,138.759029,36.915389,436.174065,10.037994,62.631938,6.211169,341.361374,15.495156
160.970195,489.712029,70.482159,309.486269,888.030604,412.655666,216.124989,47.415477,104.139145,326.462385,378.446187,1.686895,485.144327,143.668518,27.168148,309.715497,149.661493,66.415878,15.001753,539.087409,12.638678
419.907137,216.625219,487.88786,253.704462,323.226862,65.744463,271.811469,527.726782,129.805782,168.429679,637.944633,0.948507,365.946758,72.337904,36.232169,302.772338,186.944884,106.514846,3.443809,364.341969,17.275102


In [5]:
fig = px.histogram(
    x=predictions,
    nbins=30,
    title='Distribución de predicciones — Blind Test',
    labels={'x': 'target_predicted', 'y': 'Frecuencia'},
    color_discrete_sequence=['#636EFA'],
)
fig.update_layout(height=400)
fig.show()


## 3. Guardado de predicciones

In [6]:
df_output = pl.DataFrame({'target_predicted': predictions})

df_output.write_csv(OUTPUT_PATH)

print(f'Predicciones guardadas en : {OUTPUT_PATH}')
print(f'Filas                     : {len(df_output)}')
df_output.head()


Predicciones guardadas en : data/predictions.csv
Filas                     : 200


target_predicted
f64
13.025063
11.719347
15.495156
12.638678
17.275102
